# 02 • Premier pipeline tabulaire, PyTorch et Keras

`[MÉTA | Formation 4-024 | Niveau Application | TP 02 | Mode CPU local]`

**Objectif :** Construire un modèle simple avec une baseline et un split sans fuite.

**Temps indicatif :** TP principal réparti sur J1. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 00 et 01.

**Preuves de réussite :** Split disjoint, baseline, deux API, courbes, sauvegarde rechargée.

**Sources :** R01 à R04, R23.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [ ]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


## 1. Contrat de données
2 400 dossiers entièrement artificiels. Les entrées v00 à v19 ne représentent aucun critère fiscal. La cible examen_utile n’est pas un constat de fraude. resultat_apres_examen est une fuite volontaire ; dossier_id est un identifiant. Ces deux colonnes sont exclues.

In [ ]:
# EXERCICE À COMPLÉTER
# Séparer les indices avant de créer StandardScaler. Ne jamais fit sur tout X.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 2. Baselines
La référence linéaire et les arbres sont comparés sur la même validation. Le test reste fermé. Le score majoritaire seul ne détecte pas les positifs.

In [ ]:
lineaire=LogisticRegression(max_iter=300).fit(Xtrain,ytrain)
arbres=HistGradientBoostingClassifier(max_iter=80,random_state=42).fit(Xtrain,ytrain)
comparaison={"logistique":binary_metrics(yval,lineaire.predict_proba(Xval)[:,1]),"arbres":binary_metrics(yval,arbres.predict_proba(Xval)[:,1])}
print(pd.DataFrame(comparaison).T)

## 3. Réseau PyTorch et boucle entièrement visible
La sortie est un logit. BCEWithLogitsLoss combine le traitement approprié du logit et la perte binaire. La sigmoïde est utilisée pour produire les scores à l’évaluation. La validation ne reçoit aucune mise à jour.

In [ ]:
# EXERCICE À COMPLÉTER
# Exposer dans cet ordre : zero_grad, forward, perte, backward, step. model.eval() pour la validation.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 4. Le même problème avec Keras
Keras réduit le code de boucle avec compile et fit. Le moteur actif est imprimé. Cette expérience montre deux niveaux d’API, pas une comparaison de vitesse scientifiquement contrôlée. Les initialisations et détails d’optimisation ne sont pas identiques.

In [ ]:
# EXERCICE À COMPLÉTER
# Définir Input(20), deux couches cachées et une sortie sigmoïde. Conserver le même split.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 5. Variante native TensorFlow
Cette cellule n’est exécutée que si TensorFlow est installé. Son statut est écrit dans le rapport. Elle sert à reconnaître GradientTape et apply_gradients, pas à multiplier artificiellement les exercices.

In [ ]:
import importlib.util
statut_tf="non exécuté, TensorFlow absent"
if importlib.util.find_spec("tensorflow"):
    import tensorflow as tf
    tf.random.set_seed(42)
    tm=tf.keras.Sequential([tf.keras.layers.Input((20,)),tf.keras.layers.Dense(16,activation="relu"),tf.keras.layers.Dense(1)])
    opt=tf.keras.optimizers.Adam(.003)
    tx=tf.convert_to_tensor(Xtrain[:64]);ty=tf.convert_to_tensor(ytrain[:64,None],dtype=tf.float32)
    with tf.GradientTape() as tape:
        logits=tm(tx,training=True)
        tf_loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=ty,logits=logits))
    gradients=tape.gradient(tf_loss,tm.trainable_variables)
    opt.apply_gradients(zip(gradients,tm.trainable_variables))
    statut_tf=f"une étape exécutée, perte {float(tf_loss.numpy()):.4f}"
print("Branche TensorFlow :",statut_tf)

## 6. Sauvegarde PyTorch et fiche d’expérience
La comparaison porte sur les sorties d’un nouvel objet modèle. Le fichier du scaler doit rester associé aux poids. Ici, on conserve ses paramètres sous forme numérique pour le prototype. Ne charger que des fichiers maîtrisés.

In [ ]:
torch.save(modele.state_dict(),RESULTS/"02_mlp.pt")
nouveau=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
nouveau.load_state_dict(torch.load(RESULTS/"02_mlp.pt",weights_only=True));nouveau.eval()
with torch.no_grad():np.testing.assert_allclose(modele(torch.tensor(Xval[:10])).numpy(),nouveau(torch.tensor(Xval[:10])).numpy(),atol=1e-6)
np.savez(RESULTS/"02_pretraitement.npz",mean=scaler.mean_,scale=scaler.scale_)
save_result("02_comparaison",{"validation":comparaison,"tensorflow":statut_tf,"test_utilise":False})

## 7. Interpréter sans surpromettre
Rédiger : modèle préféré selon quelle métrique ; défaut principal ; expérience suivante. Ne pas choisir un vainqueur universel à partir de ce jeu artificiel. **Remédiation :** vérifier le split et le type de cible avant de changer la largeur du réseau. **Extension :** reprendre une expérience avec un taux différent en conservant les autres décisions.